<a href="https://colab.research.google.com/github/Aparna0905/Gen_Ai_Monday_Lab/blob/main/Lab11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets accelerate -q

In [2]:
import torch, math
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    set_seed
)
from datasets import Dataset

set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [3]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [4]:
def generate_text(model, tokenizer, prompt, max_length=100):
    model.eval()
    inputs = tokenizer.encode(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        output = model.generate(
            inputs,
            max_length=max_length,
            temperature=0.8,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [5]:
review_prompts = [
    "This product is",
    "I bought this phone and",
    "The quality of this item"
]

print("=== BASELINE REVIEWS ===\n")
baseline = {}

for p in review_prompts:
    baseline[p] = generate_text(model, tokenizer, p)
    print(f"Prompt: {p}\nOutput: {baseline[p]}\n")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


=== BASELINE REVIEWS ===

Prompt: This product is
Output: This product is designed to be used in conjunction with any other device and is intended for use in conjunction with any other electronic device.

The following devices may be inserted into any such device, even if the device does not have a user interface or an open device configuration:

a) any personal computer;

b) other personal computer or computer-based mobile device;

c) any computer-based home computer; and

d) any personal computer-based network-

Prompt: I bought this phone and
Output: I bought this phone and started working on it. I'm quite pleased with the way it performs, but this phone has the same design as the old Nexus 5. It's quite a bit thicker, and has a slightly better sound quality on the front touch panel than you'd expect from the Nexus 5. The other thing I like about the Nexus 5 is that it's a much smaller phone, which is nice, because it's smaller and there are no batteries in the back. I bought this p

In [7]:
corpus = [
    "this phone has an amazing battery life and the camera quality is outstanding for the price.",
    "i bought this laptop for college and it handles all my assignments and coding projects perfectly.",
    "the sound quality of these headphones is incredible with deep bass and clear vocals.",
    "this smartwatch tracks my steps accurately and the heart rate monitor is very reliable.",
    "great wireless earbuds with noise cancellation that blocks out all background sound.",
    "the keyboard feels very comfortable for long typing sessions and the backlight is a nice touch.",
    "this portable charger saved me during travel and it charges my phone three times on a single charge.",
    "the tablet screen is bright and colorful which makes watching movies a great experience.",
    "i love this fitness tracker because it motivates me to reach my daily exercise goals.",
    "this bluetooth speaker is compact but delivers surprisingly loud and clear audio.",
    "the delivery was fast and the product was packed securely with no damage at all.",
    "excellent value for money and the build quality feels premium despite the affordable price.",
    "the customer service team was very helpful when i had questions about the product features.",
    "this camera takes stunning photos in low light and the video recording quality is very smooth.",
    "i have been using this product for three months and it still works perfectly like day one.",
    "the design is sleek and modern and it looks great on my desk next to my other gadgets.",
    "easy to set up right out of the box and the instructions were clear and simple to follow.",
    "highly recommend this product to anyone looking for quality and reliability at a fair price.",
    "the software updates keep adding new features which makes this purchase even more worthwhile.",
    "best purchase i made this year and i would definitely buy from this brand again."
]

dataset = Dataset.from_dict({"text": corpus})

In [8]:
tokenized = dataset.map(
    lambda x: tokenizer(
        x["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    ),
    batched=True,
    remove_columns=["text"]
)

split = tokenized.train_test_split(test_size=0.15, seed=42)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [10]:
training_args = TrainingArguments(
    output_dir="./gpt2-reviews",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="no",
    fp16=torch.cuda.is_available()
)

In [11]:
tokenizer2 = GPT2Tokenizer.from_pretrained('gpt2')
model2 = GPT2LMHeadModel.from_pretrained('gpt2').to(device)

tokenizer2.pad_token = tokenizer2.eos_token
model2.config.pad_token_id = tokenizer2.eos_token_id

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
recipe_prompts = [
    "To make butter chicken",
    "For pasta carbonara",
    "To prepare chocolate cake"
]

baseline2 = {}

print("=== BASELINE RECIPES ===\n")

for p in recipe_prompts:
    baseline2[p] = generate_text(model2, tokenizer2, p)
    print(f"Prompt: {p}\nOutput: {baseline2[p]}\n")

=== BASELINE RECIPES ===

Prompt: To make butter chicken
Output: To make butter chicken, add ½ cup of butter and salt to a large saucepan over medium heat. Cook until bubbling and bubbling, about 30 minutes, then place in the refrigerator.

You can see how easy it is to make this recipe in the pictures in the recipe, check it out, let me know if you have any questions about it 🙂

I hope everyone is enjoying these delicious chicken wings. As an added bonus, I am all about making these amazing wings in

Prompt: For pasta carbonara
Output: For pasta carbonara with Parmesan. This was also an easy option for me to make. I was so excited to try this.

The last two meals of the day were pretty good, but the last few came with a small amount of leftover pasta. I was so excited to grab a few of these from the fridge from our local grocery store and make them for myself as a snack.

These were great for a quick meal, and it was really easy to prepare. The sauce

Prompt: To prepare chocolate cake

In [13]:
recipes = [
    "to make butter chicken start by marinating chicken pieces in yogurt with turmeric chili powder and garam masala for one hour.",
    "heat butter in a pan and fry onions until golden brown then add ginger garlic paste and cook for two minutes.",
    "add tomato puree and cook on low heat for ten minutes until the oil separates from the masala.",
    "add the marinated chicken and cook on medium heat for fifteen minutes until fully cooked.",
    "finish with fresh cream and kasuri methi and serve hot with naan or steamed rice.",
    "for pasta carbonara boil spaghetti in salted water until al dente and reserve half cup of pasta water.",
    "fry diced pancetta in olive oil until crispy and set aside.",
    "whisk together egg yolks parmesan cheese and black pepper in a bowl.",
    "toss the hot pasta with pancetta and remove from heat then quickly stir in the egg mixture.",
    "the residual heat will cook the eggs into a creamy sauce and serve immediately with extra parmesan.",
    "to prepare vegetable stir fry heat sesame oil in a wok on high heat.",
    "add sliced bell peppers broccoli florets and snap peas and toss for three minutes.",
    "pour in soy sauce oyster sauce and a pinch of sugar and stir well.",
    "add minced garlic and ginger and cook for one more minute until fragrant.",
    "serve the stir fry over steamed jasmine rice and garnish with sesame seeds.",
    "for chocolate chip cookies cream together butter and sugar until light and fluffy.",
    "beat in eggs one at a time then add vanilla extract and mix well.",
    "fold in flour baking soda and salt then gently stir in chocolate chips.",
    "scoop dough onto a baking sheet and bake at 180 degrees for twelve minutes until golden.",
    "let cookies cool on the tray for five minutes before transferring to a wire rack."
]

dataset2 = Dataset.from_dict({"text": recipes})

In [17]:
def tokenize2(x):
    return tokenizer2(x["text"], padding="max_length", truncation=True, max_length=64)

In [18]:
tokenized2 = dataset2.map(tokenize2, batched=True, remove_columns=["text"])

collator2 = DataCollatorForLanguageModeling(tokenizer=tokenizer2, mlm=False)

args2 = TrainingArguments(
    output_dir="./recipes",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    logging_steps=5,
    save_strategy="no"
)

trainer2 = Trainer(
    model=model2,
    args=args2,
    train_dataset=tokenized2,
    data_collator=collator2
)

trainer2.train()

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,3.649719
10,3.320011
15,2.670194
20,2.279820
25,2.175288
30,1.803832
35,1.706203
40,1.885350
45,1.876118
50,1.590924


TrainOutput(global_step=50, training_loss=2.295745859146118, metrics={'train_runtime': 5.5793, 'train_samples_per_second': 17.923, 'train_steps_per_second': 8.962, 'total_flos': 3266150400000.0, 'train_loss': 2.295745859146118, 'epoch': 5.0})